# 26 · More from the history file (E59-E60)

- **E59** forecast error by gap and by age of the first check; training on every pair of checks;
- **E60** forecast later likes from early likes (no views, the FB/IG situation) and chain them into the now-cast model.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
FIG = ROOT / "figures"; plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 220, "display.max_columns", 30)
import eda_utils as eu, extra_models as em, history as H, history_more as hm

In [2]:
d = H.load(ROOT); s = H.snapshots(d); pp = hm.all_pairs(s)
pp.groupby("platform").agg(pairs=("mention_id", "size"), posts=("mention_id", "nunique"), first_last=("first_last", "sum"))

,pairs,posts,first_last
platform,,,
tiktok,1773,336,336
twitter,1156,331,331
youtube,289,289,289


## E59 · Horizon curve

In [3]:
P = hm.horizon(pp); S = H.summarise(P[P.first_last]); S.round(3)

,platform,model,posts,MAE,CI low,CI high,bias
0,tiktok,persistence,336,0.248,0.209,0.287,-0.248
1,tiktok,"CatBoost, first-last pairs",336,0.164,0.132,0.196,-0.060
2,tiktok,"CatBoost, all pairs",336,0.184,0.151,0.214,-0.133
3,twitter,persistence,331,0.481,0.418,0.547,-0.481
4,twitter,"CatBoost, first-last pairs",331,0.275,0.240,0.305,-0.053
5,twitter,"CatBoost, all pairs",331,0.267,0.232,0.297,-0.077
6,youtube,persistence,289,0.352,0.279,0.439,-0.352
7,youtube,"CatBoost, first-last pairs",289,0.188,0.150,0.224,-0.027
8,youtube,"CatBoost, all pairs",289,0.188,0.150,0.224,-0.027


In [4]:
G = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P[P.first_last], p, "CatBoost, first-last pairs", k)))} for p in P.platform.unique() for k in ["CatBoost, all pairs", "persistence"]]); G.round(3)

,platform,model,gain,CI low,CI high
0,tiktok,"CatBoost, all pairs",-0.020,-0.032,-0.008
1,tiktok,persistence,-0.084,-0.110,-0.063
2,twitter,"CatBoost, all pairs",0.008,-0.002,0.018
3,twitter,persistence,-0.206,-0.256,-0.149
4,youtube,"CatBoost, all pairs",0.000,0.000,0.000
5,youtube,persistence,-0.165,-0.228,-0.111


In [5]:
Bg = hm.by_bin(P, "gap"); Ba = hm.by_bin(P, "age0")
Bg.pivot_table(index=["platform", "gap"], columns="model", values=["pairs", "MAE"], sort=False).round(3)

pairs                                                        MAE                                       
model            CatBoost, all pairs CatBoost, first-last pairs persistence CatBoost, all pairs CatBoost, first-last pairs persistence
platform gap                                                                                                                          
tiktok   12-48 h                29.0                       29.0        29.0               0.153                      0.143       0.213
         2-7 d                  87.0                       87.0        87.0               0.176                      0.152       0.223
         7-30 d                203.0                      203.0       203.0               0.198                      0.178       0.275
         <12 h                  12.0                       12.0        12.0               0.103                      0.098       0.094
         >30 d                   5.0                        5.0         5.0               0.093                      0.087       0.152
twitter  12-48 h                33.0                       33.0        33.0               0.302                      0.312       0.523
         7-30 d                  6.0                        6.0         6.0               0.191                      0.203       0.536
         <12 h                 292.0                      292.0       292.0               0.265                      0.273       0.475
youtube  12-48 h                 8.0                        8.0         8.0               0.165                      0.165       0.585
         2-7 d                  10.0                       10.0        10.0               0.427                      0.427       0.499
         7-30 d                193.0                      193.0       193.0               0.212                      0.212       0.400
         <12 h                  37.0                       37.0        37.0               0.035                      0.035       0.020
         >30 d                  41.0                       41.0        41.0               0.156                      0.156       0.345

In [6]:
Ba.pivot_table(index=["platform", "age0"], columns="model", values=["pairs", "MAE"], sort=False).round(3)

pairs                                                        MAE                                       
model            CatBoost, all pairs CatBoost, first-last pairs persistence CatBoost, all pairs CatBoost, first-last pairs persistence
platform age0                                                                                                                         
tiktok   12-48 h               144.0                      144.0       144.0               0.178                      0.155       0.226
         2-7 d                  51.0                       51.0        51.0               0.168                      0.174       0.209
         7-30 d                 87.0                       87.0        87.0               0.136                      0.145       0.153
         <12 h                  46.0                       46.0        46.0               0.325                      0.228       0.574
         >30 d                   8.0                        8.0         8.0               0.094                      0.090       0.068
twitter  12-48 h                13.0                       13.0        13.0               0.048                      0.112       0.038
         2-7 d                   1.0                        1.0         1.0               0.010                      0.021       0.000
         7-30 d                  1.0                        1.0         1.0               0.050                      0.108       0.000
         <12 h                 316.0                      316.0       316.0               0.278                      0.283       0.502
youtube  12-48 h                26.0                       26.0        26.0               0.377                      0.377       0.813
         2-7 d                  78.0                       78.0        78.0               0.224                      0.224       0.283
         7-30 d                 47.0                       47.0        47.0               0.113                      0.113       0.127
         <12 h                  47.0                       47.0        47.0               0.428                      0.428       1.102
         >30 d                  91.0                       91.0        91.0               0.017                      0.017       0.009

## E60 · Early reactions without views

In [7]:
f, meta = em.load(ROOT); p1 = H.pairs(s)
Lk, V = hm.early_reactions(p1, f, meta)
SL = H.summarise(Lk); SV = H.summarise(V); pd.concat([SL.assign(target="log likes, last check"), SV.assign(target="log views, last check")]).round(3)

,platform,model,posts,MAE,CI low,CI high,bias,target
0,tiktok,likes: persistence,331,0.230,0.190,0.272,-0.230,"log likes, last check"
1,tiktok,likes: CatBoost,331,0.162,0.133,0.190,-0.055,"log likes, last check"
2,twitter,likes: persistence,282,0.147,0.106,0.198,-0.147,"log likes, last check"
3,twitter,likes: CatBoost,282,0.149,0.115,0.185,-0.063,"log likes, last check"
0,tiktok,E41: CatBoost reactions + ages,331,0.365,0.322,0.406,-0.042,"log views, last check"
1,tiktok,chain: forecast reactions -> now-cast,331,0.332,0.296,0.369,-0.074,"log views, last check"
2,tiktok,chain with true later reactions (upper bound),331,0.289,0.259,0.323,-0.037,"log views, last check"
3,twitter,E41: CatBoost reactions + ages,282,0.525,0.473,0.581,-0.062,"log views, last check"
4,twitter,chain: forecast reactions -> now-cast,282,0.492,0.436,0.552,-0.109,"log views, last check"
5,twitter,chain with true later reactions (upper bound),282,0.415,0.374,0.462,-0.036,"log views, last check"


In [8]:
GE = pd.DataFrame([{"platform": p, "base": a, "model": b, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(Q, p, a, b)))} for p in V.platform.unique() for Q, a, b in [(Lk, "likes: persistence", "likes: CatBoost"), (V, "E41: CatBoost reactions + ages", "chain: forecast reactions -> now-cast"), (V, "E41: CatBoost reactions + ages", "chain with true later reactions (upper bound)")]]); GE.round(3)

,platform,base,model,gain,CI low,CI high
0,tiktok,likes: persistence,likes: CatBoost,0.068,0.044,0.097
1,tiktok,E41: CatBoost reactions + ages,chain: forecast reactions -> now-cast,0.034,0.012,0.058
2,tiktok,E41: CatBoost reactions + ages,chain with true later reactions (upper bound),0.076,0.045,0.104
3,twitter,likes: persistence,likes: CatBoost,-0.001,-0.020,0.018
4,twitter,E41: CatBoost reactions + ages,chain: forecast reactions -> now-cast,0.033,-0.010,0.077
5,twitter,E41: CatBoost reactions + ages,chain with true later reactions (upper bound),0.110,0.057,0.165


## Takeaways
See `docs/en/experiments.md` (E59-E60). Numbers: `data/processed/history_more_numbers.json`.

In [9]:
json.dump({"pairs": pp.groupby("platform").size().to_dict(), "horizon": S.to_dict("records"), "horizon_gains": G.to_dict("records"), "by_gap": Bg.to_dict("records"), "by_age0": Ba.to_dict("records"), "likes": SL.to_dict("records"), "views": SV.to_dict("records"), "early_gains": GE.to_dict("records")},
          open(ROOT / "data" / "processed" / "history_more_numbers.json", "w"), indent=1, default=float)